In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input

/Users/taniyashuba/PycharmProjects/NeuroSymbolicDynamics/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# 1. Загрузка данных
df = pd.read_csv('/Users/taniyashuba/PycharmProjects/NeuroSymbolicDynamics/data/lorenz.csv')        # предполагается, что файл в рабочей папке
series = df['x'].values.astype('float32')

In [3]:
# 2. Подготовка выборок: из каждой длины seq_len предсказываем следующую точку
seq_len = 10
X, y = [], []
for i in range(len(series) - seq_len):
    X.append(series[i:i+seq_len])
    y.append(series[i+seq_len])
X = np.array(X)                         # (N, seq_len)
y = np.array(y)                         # (N,)

In [4]:
# 3. Разбивка на train/val
split = 4000
X_train, y_train = X[:split], y[:split]
X_val,   y_val   = X[split:split+500], y[split:split+500]

In [5]:
# 4. Приведение формы к (samples, seq_len, features)
X_train = X_train[..., None]
X_val   = X_val[...,   None]

In [6]:
# 5. Описание модели с возвратом полного тракта скрытых состояний
input_layer = Input(shape=(seq_len,1))
lstm_out   = layers.LSTM(64, return_sequences=True, name='lstm')(input_layer)
# последний выход для предсказания
pred_output = layers.Dense(1, name='pred')(lstm_out[:,-1,:])
# модель, возвращающая и предсказание, и всю траекторию скрытых состояний
model = Model(inputs=input_layer, outputs=[pred_output, lstm_out])

model.compile(
    optimizer='adam',
    loss=['mse', None],    # только по первому выходу считаем MSE
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 10, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pred (Dense)                    │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,961 (66.25 KB)

 Trainable params: 16,961 (66.25 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# 6. Обучение
model.fit(
    X_train, [y_train, np.zeros_like(y_train)],
    validation_data=(X_val, [y_val, np.zeros_like(y_val)]),
    epochs=50,
    batch_size=32
)

Epoch 1/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 27.7923 - val_loss: 5.2775
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.7483 - val_loss: 0.9801
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6890 - val_loss: 0.3551
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3303 - val_loss: 0.1698
Epoch 5/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1664 - val_loss: 0.0762
Epoch 6/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1103 - val_loss: 0.0458
Epoch 7/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0711 - val_loss: 0.0245
Epoch 8/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0365 - val_loss: 0.0203
Epoch 9/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0391 - val_loss: 0.0121
Epoch 10/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0253 - val_loss: 0.0166
Epoch 11/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0194 - val_loss: 0.0096
Epoch 12/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/ste

In [8]:
# 7. Извлечение скрытых состояний на валидации
preds, hidden_seq = model.predict(X_val)
# hidden_seq.shape = (N_val, seq_len, 64)

# Скрытое состояние в каждый момент t: hidden_seq[:, t, :]
# Например, возьмём последнее состояние каждого примера
h_last = hidden_seq[:, -1, :]            # (N_val, 64)

# --- Дальше можно сразу перейти к time-delay embedding и символической динамике ---

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [9]:
# 8. Сохраним скрытые состояния для дальнейшего анализа
import numpy as _np
_np.save('hidden_rnn.npy', h_last)

In [10]:
# Загружаем .npy-файл
hidden_states = np.load('hidden_rnn.npy')

# Проверяем тип данных и форму массива
print(f"Тип данных: {type(hidden_states)}")
print(f"Форма массива: {hidden_states.shape}")
print(f"Пример содержимого:\n{hidden_states[0]}")

Тип данных: <class 'numpy.ndarray'>
Форма массива: (500, 64)
Пример содержимого:
[ 0.1054113   0.03521043  0.36156696 -0.12140986  0.12871602 -0.4857082
 -0.08266257  0.14758006  0.12099995  0.05054994 -0.06198772 -0.1618436
  0.12946945 -0.08324792 -0.18745565 -0.10723644  0.28560027 -0.21840912
 -0.34364098 -0.03319699 -0.22587654  0.35912833  0.3024181  -0.41100177
 -0.3613578   0.16898212 -0.4048049   0.38022545 -0.4955489   0.18515888
  0.04400759 -0.22298244 -0.15320738 -0.05230178 -0.12581311 -0.13666989
 -0.3077669  -0.4320455  -0.20170043 -0.12422785  0.24617529 -0.331254
 -0.35729814  0.04717606  0.02313778  0.0983142   0.01100853  0.17359205
  0.16090423  0.01984285 -0.05383366 -0.08494695 -0.16528894 -0.13293952
  0.05169102 -0.09567972 -0.22607422 -0.3567494   0.20700717  0.0044342
 -0.02354059  0.1776596  -0.11929578  0.06380536]
